# Sample Size Planning for Persona Fairness Experiments

## Setup

We have a **3-class ordinal triage** problem: self-care < non-urgent < urgent. Each patient is evaluated under two personae (paired design). We want to detect differences in **over-triage rate** and **under-triage rate** between personae.

### Why the 3-class structure matters

Not every case is eligible for every error type:
- **Over-triage** can only happen when the gold label is *not* the highest class. Eligible cases: gold = self-care or non-urgent. Fraction eligible = P(self-care) + P(non-urgent) = 1 - P(urgent).
- **Under-triage** can only happen when the gold label is *not* the lowest class. Eligible cases: gold = non-urgent or urgent. Fraction eligible = P(non-urgent) + P(urgent) = 1 - P(self-care).

The gold class distribution therefore determines how many of your total N cases are actually usable for each metric. A distribution skewed toward urgent cases shrinks the over-triage eligible pool; one skewed toward self-care shrinks the under-triage pool.

### Paired test: McNemar's test

Since each patient appears under both personae, we use McNemar's test on the 2x2 discordance table. Among eligible pairs:
- **p12** = P(over-triaged under persona A, not under persona B)
- **p21** = P(over-triaged under persona B, not under persona A)
- **Discordant rate** = p12 + p21 (fraction of eligible pairs where the outcome flips)
- **Delta** = p12 - p21 (the paired difference we want to detect)

### What this notebook computes

Given a **total sample size N**, we solve for the **minimum detectable difference (MDD)** — the smallest delta (in percentage points) that McNemar's test can detect at 80% power, alpha = 0.05. We report MDD separately for over-triage and under-triage, since each has a different eligible fraction.

We explore a few realistic scenarios for:
1. **Gold class distributions** — determines eligible fractions
2. **Discordant rates** — the key unknown, estimated from pilot data or prior experiments

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq

In [19]:
def mcnemar_power(n, p12, p21, alpha=0.05):
    """Power of McNemar's test for given n eligible cases."""
    z_a = norm.ppf(1 - alpha / 2)
    delta = p12 - p21
    disc = p12 + p21
    denom = np.sqrt(disc - delta**2)
    if denom <= 0:
        return 1.0
    numer = np.sqrt(n) * abs(delta) - z_a * np.sqrt(disc)
    return float(norm.cdf(numer / denom))


def mdd_for_n(n_eligible, disc_rate, alpha=0.05, power=0.80):
    """Minimum detectable difference (delta) for given n eligible cases and discordant rate.
    Solves: find smallest delta such that McNemar power >= target."""
    z_a = norm.ppf(1 - alpha / 2)
    z_b = norm.ppf(power)

    def power_minus_target(delta):
        p12 = (disc_rate + delta) / 2
        p21 = (disc_rate - delta) / 2
        if p21 <= 0:
            return 1.0
        return mcnemar_power(n_eligible, p12, p21, alpha) - power

    try:
        result = brentq(power_minus_target, 1e-6, disc_rate - 1e-6)
        return result
    except ValueError:
        return np.nan

## Scenarios

In [20]:
# --- Scenarios ---

# Gold class distributions: (self-care, non-urgent, urgent)
CLASS_DISTS = {
    "Balanced (33/34/33)":        (0.33, 0.34, 0.33),
    "SC low (20/40/40)":          (0.20, 0.40, 0.40),
    "NU low (40/20/40)":          (0.40, 0.20, 0.40),
    "U low (40/40/20)":           (0.40, 0.40, 0.20),
    "Skew urgent (15/35/50)":     (0.15, 0.35, 0.50),
    "Heavy middle (20/55/25)":    (0.20, 0.55, 0.25),
}

# Discordant rates: fraction of eligible pairs where the outcome flips
DISC_RATES = [0.025, 0.05, 0.075, 0.10, 0.125, 0.15, 0.175, 0.20]

# Total sample sizes to evaluate
SAMPLE_SIZES = [200, 400, 600, 800, 1000, 1200, 1500, 2000]

ALPHA = 0.05
POWER = 0.80

# Show eligible fractions as a DataFrame
elig_rows = []
for name, (sc, nu, u) in CLASS_DISTS.items():
    elig_rows.append({
        "Distribution": name,
        "Self-care": f"{sc:.0%}",
        "Non-urgent": f"{nu:.0%}",
        "Urgent": f"{u:.0%}",
        "OT eligible": f"{1 - u:.0%}",
        "UT eligible": f"{1 - sc:.0%}",
    })
display(pd.DataFrame(elig_rows).set_index("Distribution"))

,Self-care,Non-urgent,Urgent,OT eligible,UT eligible
Distribution,,,,,
Balanced (33/34/33),33%,34%,33%,67%,67%
SC low (20/40/40),20%,40%,40%,60%,80%
NU low (40/20/40),40%,20%,40%,60%,60%
U low (40/40/20),40%,40%,20%,80%,60%
Skew urgent (15/35/50),15%,35%,50%,50%,85%
Heavy middle (20/55/25),20%,55%,25%,75%,80%


In [21]:
# --- Compute MDD for all scenarios ---

rows = []
for dist_name, (sc, nu, u) in CLASS_DISTS.items():
    ot_elig_frac = 1 - u
    ut_elig_frac = 1 - sc
    for disc in DISC_RATES:
        for n_total in SAMPLE_SIZES:
            n_ot = int(n_total * ot_elig_frac)
            n_ut = int(n_total * ut_elig_frac)
            mdd_ot = mdd_for_n(n_ot, disc) if n_ot > 10 else np.nan
            mdd_ut = mdd_for_n(n_ut, disc) if n_ut > 10 else np.nan
            rows.append({
                "distribution": dist_name,
                "disc_rate": disc,
                "n_total": n_total,
                "mdd_ot_pp": round(mdd_ot * 100, 1) if not np.isnan(mdd_ot) else np.nan,
                "mdd_ut_pp": round(mdd_ut * 100, 1) if not np.isnan(mdd_ut) else np.nan,
            })

mdd_df = pd.DataFrame(rows)
print(f"Computed {len(mdd_df)} scenarios")

Computed 384 scenarios


## MDD Tables

Each table shows one class distribution. Rows are grouped by discordant rate. For each total sample size N, we report the minimum detectable difference (in pp) for over-triage and under-triage separately.

**Reading the table**: at N=1000 cases with a 10% discordant rate and balanced classes, an MDD of 4.2pp means you can detect a 4.2 percentage-point paired difference in over-triage rate between two personae (80% power, two-sided alpha=0.05).

In [22]:
# --- Display MDD tables as styled DataFrames ---

MDD_THRESHOLD = 5.0  # pp — green rows where MDD <= this across all disc rates

def style_mdd(df):
    """Green background for rows where all MDD values <= threshold (sufficient sensitivity)."""
    def row_style(row):
        vals = []
        for v in row:
            try:
                vals.append(float(v))
            except (ValueError, TypeError):
                vals.append(np.nan)
        if vals and all(not np.isnan(v) and v <= MDD_THRESHOLD for v in vals):
            return ["background-color: #d4edda"] * len(row)
        return [""] * len(row)
    return df.style.apply(row_style, axis=1).format("{:.1f}")


for dist_name, (sc, nu, u) in CLASS_DISTS.items():
    ot_elig = 1 - u
    ut_elig = 1 - sc
    sub = mdd_df[mdd_df["distribution"] == dist_name]

    # Over-triage table: rows=N, cols=disc_rate
    piv_ot = sub.pivot_table(index="n_total", columns="disc_rate", values="mdd_ot_pp", aggfunc="first")
    piv_ot.columns = [f"{d:.1%}" for d in piv_ot.columns]
    piv_ot.columns.name = "Discordance rate"
    piv_ot.index.name = "N total"

    # Under-triage table
    piv_ut = sub.pivot_table(index="n_total", columns="disc_rate", values="mdd_ut_pp", aggfunc="first")
    piv_ut.columns = [f"{d:.1%}" for d in piv_ut.columns]
    piv_ut.columns.name = "Discordance rate"
    piv_ut.index.name = "N total"

    print(f"\n{'=' * 70}")
    print(f"  {dist_name}  |  OT eligible = {ot_elig:.0%}  |  UT eligible = {ut_elig:.0%}")
    print(f"{'=' * 70}")

    print(f"\n  Over-triage — Minimum Detectable Difference (pp):")
    display(style_mdd(piv_ot))

    print(f"\n  Under-triage — Minimum Detectable Difference (pp):")
    display(style_mdd(piv_ut))


  Balanced (33/34/33)  |  OT eligible = 67%  |  UT eligible = 67%

  Over-triage — Minimum Detectable Difference (pp):


Discordance rate,2.5%,5.0%,7.5%,10.0%,12.5%,15.0%,17.5%,20.0%
N total,,,,,,,,
200,nan,nan,6.6,7.6,8.5,9.3,10.0,10.7
400,nan,3.8,4.7,5.4,6.0,6.6,7.1,7.6
600,2.2,3.1,3.8,4.4,4.9,5.4,5.8,6.2
800,1.9,2.7,3.3,3.8,4.3,4.7,5.1,5.4
1000,1.7,2.4,3.0,3.4,3.8,4.2,4.5,4.8
1200,1.6,2.2,2.7,3.1,3.5,3.8,4.1,4.4
1500,1.4,2.0,2.4,2.8,3.1,3.4,3.7,3.9
2000,1.2,1.7,2.1,2.4,2.7,3.0,3.2,3.4



  Under-triage — Minimum Detectable Difference (pp):


Discordance rate,2.5%,5.0%,7.5%,10.0%,12.5%,15.0%,17.5%,20.0%
N total,,,,,,,,
200,nan,nan,6.6,7.6,8.5,9.3,10.0,10.7
400,nan,3.8,4.7,5.4,6.0,6.6,7.1,7.6
600,2.2,3.1,3.8,4.4,4.9,5.4,5.8,6.2
800,1.9,2.7,3.3,3.8,4.3,4.7,5.1,5.4
1000,1.7,2.4,3.0,3.4,3.8,4.2,4.5,4.8
1200,1.6,2.2,2.7,3.1,3.5,3.8,4.1,4.4
1500,1.4,2.0,2.4,2.8,3.1,3.4,3.7,3.9
2000,1.2,1.7,2.1,2.4,2.7,3.0,3.2,3.4



  SC low (20/40/40)  |  OT eligible = 60%  |  UT eligible = 80%

  Over-triage — Minimum Detectable Difference (pp):


Discordance rate,2.5%,5.0%,7.5%,10.0%,12.5%,15.0%,17.5%,20.0%
N total,,,,,,,,
200,nan,nan,6.9,8.0,9.0,9.8,10.6,11.3
400,nan,4.0,4.9,5.7,6.4,7.0,7.5,8.0
600,2.3,3.3,4.0,4.7,5.2,5.7,6.2,6.6
800,2.0,2.9,3.5,4.0,4.5,4.9,5.3,5.7
1000,1.8,2.6,3.1,3.6,4.0,4.4,4.8,5.1
1200,1.6,2.3,2.9,3.3,3.7,4.0,4.4,4.7
1500,1.5,2.1,2.6,2.9,3.3,3.6,3.9,4.2
2000,1.3,1.8,2.2,2.6,2.9,3.1,3.4,3.6


... (14 outputs left out)


  Heavy middle (20/55/25)  |  OT eligible = 75%  |  UT eligible = 80%

  Over-triage — Minimum Detectable Difference (pp):


Discordance rate,2.5%,5.0%,7.5%,10.0%,12.5%,15.0%,17.5%,20.0%
N total,,,,,,,,
200,nan,nan,6.2,7.2,8.0,8.8,9.5,10.1
400,nan,3.6,4.4,5.1,5.7,6.2,6.7,7.2
600,2.1,2.9,3.6,4.2,4.7,5.1,5.5,5.9
800,1.8,2.6,3.1,3.6,4.0,4.4,4.8,5.1
1000,1.6,2.3,2.8,3.2,3.6,4.0,4.3,4.6
1200,1.5,2.1,2.6,2.9,3.3,3.6,3.9,4.2
1500,1.3,1.9,2.3,2.6,3.0,3.2,3.5,3.7
2000,1.1,1.6,2.0,2.3,2.6,2.8,3.0,3.2



  Under-triage — Minimum Detectable Difference (pp):


Discordance rate,2.5%,5.0%,7.5%,10.0%,12.5%,15.0%,17.5%,20.0%
N total,,,,,,,,
200,nan,4.9,6.0,7.0,7.8,8.5,9.2,9.8
400,2.5,3.5,4.3,4.9,5.5,6.0,6.5,7.0
600,2.0,2.9,3.5,4.0,4.5,4.9,5.3,5.7
800,1.7,2.5,3.0,3.5,3.9,4.3,4.6,4.9
1000,1.6,2.2,2.7,3.1,3.5,3.8,4.1,4.4
1200,1.4,2.0,2.5,2.9,3.2,3.5,3.8,4.0
1500,1.3,1.8,2.2,2.6,2.9,3.1,3.4,3.6
2000,1.1,1.6,1.9,2.2,2.5,2.7,2.9,3.1


## How to use these tables

1. **Estimate your class distribution** from the dataset(s) you plan to use. This determines the eligible fraction for each metric.

2. **Estimate the discordant rate** from a pilot run (e.g., 50-100 cases under two personae). Count the fraction of eligible pairs where the binary outcome flips. If no pilot is available, 8-12% is a reasonable starting range based on our experiments.

3. **Pick your budget N** and read the MDD. If the MDD is larger than the effect you care about, increase N or accept lower sensitivity.

4. **Plan for the harder metric.** Over-triage and under-triage have different eligible fractions and potentially different discordant rates. The metric with the higher MDD is the bottleneck — size for that one.